# FotMob Fixtures Scraper v2 — All Domestic Leagues

Season format types:
- **`standard`** — UEFA-style split year: `2025/2026`
- **`calendar`** — Single calendar year: `2025`, `2026`
- **`split_standard`** — Apertura/Clausura on split year: `2025/2026 - Apertura`, `2025/2026 - Clausura` (Mexico, Honduras)
- **`split_calendar`** — Apertura/Clausura on calendar year: `2025 - Apertura`, `2025 - Clausura` (Colombia)

To scrape different seasons, just change `START_YEAR` and `END_YEAR`.

In [3]:
import requests
import json
import time
import os

# --- Year range ---
# For standard:       generates 2023/2024, 2024/2025, 2025/2026
# For calendar:       generates 2023, 2024, 2025, 2026
# For split_standard: generates 2023/2024 - Apertura, 2023/2024 - Clausura, etc.
# For split_calendar: generates 2023 - Apertura, 2023 - Clausura, etc.
START_YEAR = 2023
END_YEAR   = 2025

# --- League config ---
# format options: standard, calendar, split_standard, split_calendar
LEAGUES = {
    # England
    "eng_47":   {"id": 47,   "name": "Premier League",          "country": "England",      "format": "standard"},
    # "eng_48":   {"id": 48,   "name": "Championship",            "country": "England",      "format": "standard"},
    # Germany
    # "ger_54":   {"id": 54,   "name": "Bundesliga",              "country": "Germany",      "format": "standard"},
    # France
    # "fra_53":   {"id": 53,   "name": "Ligue 1",                 "country": "France",       "format": "standard"},
    # Spain
    # "esp_87":   {"id": 87,   "name": "LaLiga",                  "country": "Spain",        "format": "standard"},
    # Italy
    # "ita_55":   {"id": 55,   "name": "Serie A",                 "country": "Italy",        "format": "standard"},
    # Netherlands
    # "ned_57":   {"id": 57,   "name": "Eredivisie",              "country": "Netherlands",  "format": "standard"},
    # Portugal
    # "por_61":   {"id": 61,   "name": "Liga Portugal",           "country": "Portugal",     "format": "standard"},
    # Belgium
    # "bel_40":   {"id": 40,   "name": "First Division A",        "country": "Belgium",      "format": "standard"},
    # Scotland
    # "sco_64":   {"id": 64,   "name": "Premiership",             "country": "Scotland",     "format": "standard"},
    # Wales
    # "wal_116":  {"id": 116,  "name": "Cymru Premier",           "country": "Wales",        "format": "standard"},
    # Greece
    # "gre_135":  {"id": 135,  "name": "Super League 1",          "country": "Greece",       "format": "standard"},
    # Cyprus
    # "cyp_136":  {"id": 136,  "name": "1. Division",             "country": "Cyprus",       "format": "standard"},
    # Croatia
    # "cro_252":  {"id": 252,  "name": "HNL",                     "country": "Croatia",      "format": "standard"},
    # Czechia
    # "cze_122":  {"id": 122,  "name": "1. Liga",                 "country": "Czechia",      "format": "standard"},
    # Slovakia
    # "svk_176":  {"id": 176,  "name": "1. liga",                 "country": "Slovakia",     "format": "standard"},
    # Slovenia
    # "svn_173":  {"id": 173,  "name": "Prva Liga",               "country": "Slovenia",     "format": "standard"},
    # Poland
    # "pol_196":  {"id": 196,  "name": "Ekstraklasa",             "country": "Poland",       "format": "standard"},
    # Hungary
    # "hun_212":  {"id": 212,  "name": "Nemzeti Bajnokság I",     "country": "Hungary",      "format": "standard"},
    # Romania
    # "rou_189":  {"id": 189,  "name": "Liga I",                  "country": "Romania",      "format": "standard"},
    # Serbia
    # "srb_182":  {"id": 182,  "name": "Super Liga",              "country": "Serbia",       "format": "standard"},
    # Bulgaria
    # "bul_270":  {"id": 270,  "name": "First Professional League","country": "Bulgaria",    "format": "standard"},
    # Switzerland
    # "sui_69":   {"id": 69,   "name": "Super League",            "country": "Switzerland",  "format": "standard"},
    # Austria
    # "aut_38":   {"id": 38,   "name": "Bundesliga",              "country": "Austria",      "format": "standard"},
    # Denmark
    # "den_46":   {"id": 46,   "name": "Superligaen",             "country": "Denmark",      "format": "standard"},
    # Turkey
    # "tur_71":   {"id": 71,   "name": "Süper Lig",               "country": "Turkey",       "format": "standard"},
    # Russia
    # "rus_63":   {"id": 63,   "name": "Premier League",          "country": "Russia",       "format": "standard"},
    # Armenia
    # "arm_118":  {"id": 118,  "name": "Premier League",          "country": "Armenia",      "format": "standard"},
    # Israel
    # "isr_127":  {"id": 127,  "name": "Ligat ha'Al",             "country": "Israel",       "format": "standard"},
    # Saudi Arabia
    # "ksa_536":  {"id": 536,  "name": "Saudi Pro League",        "country": "Saudi Arabia", "format": "standard"},
    # Qatar
    # "qat_535":  {"id": 535,  "name": "Qatar Stars League",      "country": "Qatar",        "format": "standard"},
    # UAE
    # "uae_538":  {"id": 538,  "name": "Pro League",              "country": "UAE",          "format": "standard"},
    # Iran
    # "irn_523":  {"id": 523,  "name": "Persian Gulf",            "country": "Iran",         "format": "standard"},
    # Iraq
    # "irq_524":  {"id": 524,  "name": "Stars League",            "country": "Iraq",         "format": "standard"},
    # Egypt
    # "egy_519":  {"id": 519,  "name": "Premier League",          "country": "Egypt",        "format": "standard"},
    # Morocco
    # "mar_530":  {"id": 530,  "name": "Botola Pro",              "country": "Morocco",      "format": "standard"},
    # Tunisia
    # "tun_544":  {"id": 544,  "name": "Ligue I",                 "country": "Tunisia",      "format": "standard"},
    # Algeria
    # "alg_516":  {"id": 516,  "name": "Ligue 1",                 "country": "Algeria",      "format": "standard"},
    # Ghana
    # "gha_522":  {"id": 522,  "name": "Premier League",          "country": "Ghana",        "format": "standard"},
    # South Africa
    # "rsa_537":  {"id": 537,  "name": "Premier Soccer League",   "country": "South Africa", "format": "standard"},
    # Australia — standard (2025/2026 style)
    # "aus_113":  {"id": 113,  "name": "A-League",                "country": "Australia",    "format": "standard"},
    # Indonesia — standard
    # "idn_8983": {"id": 8983, "name": "Super League",            "country": "Indonesia",    "format": "standard"},
    # Malaysia — standard
    # "mas_8985": {"id": 8985, "name": "Liga Super",              "country": "Malaysia",     "format": "standard"},
    # Thailand — standard
    # "tha_8984": {"id": 8984, "name": "Thai League",             "country": "Thailand",     "format": "standard"},
    # Costa Rica — standard
    # "crc_121":  {"id": 121,  "name": "Primera Division",        "country": "Costa Rica",   "format": "standard"},
    # Finland — calendar (summer league)
    # "fin_51":   {"id": 51,   "name": "Veikkausliiga",           "country": "Finland",      "format": "calendar"},
    # Norway — calendar
    # "nor_59":   {"id": 59,   "name": "Eliteserien",             "country": "Norway",       "format": "calendar"},
    # Sweden — calendar
    # "swe_67":   {"id": 67,   "name": "Allsvenskan",             "country": "Sweden",       "format": "calendar"},
    # Ireland — calendar
    # "irl_126":  {"id": 126,  "name": "Premier Division",        "country": "Ireland",      "format": "calendar"},
    # Kazakhstan — calendar
    # "kaz_225":  {"id": 225,  "name": "Premier League",          "country": "Kazakhstan",   "format": "calendar"},
    # Brazil
    # "bra_268":  {"id": 268,  "name": "Serie A",                 "country": "Brazil",       "format": "calendar"},
    # Argentina
    # "arg_112":  {"id": 112,  "name": "Liga Profesional",        "country": "Argentina",    "format": "calendar"},
    # Chile
    # "chi_273":  {"id": 273,  "name": "Liga de Primera",         "country": "Chile",        "format": "calendar"},
    # Ecuador
    # "ecu_246":  {"id": 246,  "name": "Serie A",                 "country": "Ecuador",      "format": "calendar"},
    # Paraguay
    # "par_199":  {"id": 199,  "name": "Division Profesional",    "country": "Paraguay",     "format": "calendar"},
    # Venezuela
    # "ven_339":  {"id": 339,  "name": "Primera Division",        "country": "Venezuela",    "format": "calendar"},
    # Panama
    # "pan_9039": {"id": 9039, "name": "LPF",                     "country": "Panama",       "format": "calendar"},
    # USA
    # "usa_130":  {"id": 130,  "name": "MLS",                     "country": "USA",          "format": "calendar"},
    # Japan
    # "jpn_223":  {"id": 223,  "name": "J. League",               "country": "Japan",        "format": "calendar"},
    # South Korea
    # "kor_9080": {"id": 9080, "name": "K League 1",              "country": "South Korea",  "format": "calendar"},
    # China
    # "chn_120":  {"id": 120,  "name": "Super League",            "country": "China",        "format": "calendar"},
    # Uzbekistan
    # "uzb_540":  {"id": 540,  "name": "Superliga",               "country": "Uzbekistan",   "format": "calendar"},
    # New Zealand
    # "nzl_8870": {"id": 8870, "name": "Championship",            "country": "New Zealand",  "format": "calendar"},
    # Mexico — split_standard (YYYY/YYYY+1 - Apertura/Clausura)
    # "mex_230":  {"id": 230,  "name": "Liga MX",                 "country": "Mexico",       "format": "split_standard"},
    # Honduras — split_standard
    # "hon_337":  {"id": 337,  "name": "Liga Nacional",           "country": "Honduras",     "format": "split_standard"},
    # Colombia — split_calendar (YYYY - Apertura/Clausura)
    # "col_274":  {"id": 274,  "name": "Primera A",               "country": "Colombia",     "format": "split_calendar"},
}

# --- Season string generators ---
def get_seasons(fmt, start_year, end_year):
    seasons = []
    if fmt == "standard":
        for y in range(start_year, end_year):
            seasons.append(f"{y}/{y+1}")
    elif fmt == "calendar":
        for y in range(start_year, end_year + 1):
            seasons.append(str(y))
    elif fmt == "split_standard":
        for y in range(start_year, end_year):
            seasons.append(f"{y}/{y+1} - Apertura")
            seasons.append(f"{y}/{y+1} - Clausura")
    elif fmt == "split_calendar":
        for y in range(start_year, end_year + 1):
            seasons.append(f"{y} - Apertura")
            seasons.append(f"{y} - Clausura")
    return seasons

OUT_DIR = "fixtures_tiredness"
os.makedirs(OUT_DIR, exist_ok=True)
headers = {
    "Referer": "https://www.fotmob.com/leagues/47/overview/premier-league?season=2023-2024",
    "Sec-Ch-Ua": '"Chromium";v="148", "Google Chrome";v="148", "Not/A)Brand";v="99"',
    "Sec-Ch-Ua-Mobile": "?0",
    "Sec-Ch-Ua-Platform": '"macOS"',
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36",
    "X-Mas": "eyJib2R5Ijp7InVybCI6Ii9hcGkvZGF0YS9sZWFndWVzP2lkPTQ3JmNjb2RlMz1VU0FfQ0Emc2Vhc29uPTIwMjMlMkYyMDI0IiwiY29kZSI6MTc4MDAwNjQ4NTU0MiwiZm9vIjoicHJvZHVjdGlvbjoyMzdlMjI5OWM4MjI3ZDgxNWE5ZDNjZjM4Yjc3NmU1NmU5OGNiMWQwIn0sInNpZ25hdHVyZSI6IjI1QTU1NDJFMTEyRjZDOUY5RTQxMzczQzBCM0I2QTc5In0=",
}

print("Config loaded.")
print(f"Year range: {START_YEAR} → {END_YEAR}")
print(f"Output dir: {OUT_DIR}/")

Config loaded.
Year range: 2023 → 2025
Output dir: fixtures_tiredness/


In [7]:
for key, league in LEAGUES.items():
    league_id    = league["id"]
    league_name  = league["name"]
    country      = league["country"]
    fmt          = league["format"]
    seasons      = get_seasons(fmt, START_YEAR, END_YEAR)

    for season in seasons:
        season_slug = season.replace("/", "_").replace(" ", "_").replace("-", "-")
        filename    = f"{OUT_DIR}/{key}_{season_slug}_fixtures.json"

        if os.path.exists(filename):
            print(f"  [skip] {country} — {league_name} {season}")
            continue

        print(f"Fetching {country} — {league_name} {season}...", end=" ", flush=True)

        try:
            r = requests.get(
                f"https://www.fotmob.com/api/data/leagues?id={league_id}&ccode3=USA_NY&season={requests.utils.quote(season)}",
                headers=headers,
                timeout=10
            )

            if r.status_code != 200:
                print(f"✗ HTTP {r.status_code}")
                continue

            data         = r.json()
            sel_season   = data.get("details", {}).get("selectedSeason", "")
            all_matches  = data.get("fixtures", {}).get("allMatches", [])
            finished     = [m for m in all_matches if m["status"].get("finished")]
            upcoming     = [m for m in all_matches if not m["status"].get("finished")]
            mismatch_flag = " ⚠️ MISMATCH" if sel_season != season else ""

            print(f"✓  selected='{sel_season}'  {len(finished)} fin | {len(upcoming)} up{mismatch_flag}")

            output = {
                "meta": {
                    "league_key":       key,
                    "league_id":        league_id,
                    "league_name":      league_name,
                    "country":          country,
                    "format":           fmt,
                    "requested_season": season,
                    "selected_season":  sel_season,
                    "finished_count":   len(finished),
                    "upcoming_count":   len(upcoming),
                },
                "data": data
            }

            with open(filename, "w") as f:
                json.dump(output, f)

        except Exception as e:
            print(f"✗ Error: {e}")

        time.sleep(0.3)

print("\nDone.")


#the seasons in the past are harder to get.

  [skip] England — Premier League 2023/2024
Fetching England — Premier League 2024/2025... ✓  selected='2024/2025'  380 fin | 0 up

Done.
